# NeMo MSDD vs Pyannote 3.1 — Diarization Comparison

Runs the same audio through both diarizers using a shared Whisper transcription (large-v3).
Compares accuracy by measuring speaker assignment agreement and optional YouTube caption alignment.

**Requirements:**
- Colab Pro (A100 recommended)
- HuggingFace token (accept Pyannote 3.1 license at https://huggingface.co/pyannote/speaker-diarization-3.1)

## 1. Setup

In [ ]:
# Clone the repo
!git clone --branch feat/people-agents https://github.com/cha7ura/vault.git /content/vault 2>/dev/null || echo 'Already cloned'
%cd /content/vault/vendor/whisper-diarization

In [ ]:
# Install dependencies
!pip install -q "faster-whisper>=1.1.0"
!pip install -q "nemo-toolkit[asr]>=2.5.0"
!pip install -q "pyannote.audio>=3.1.0"
!pip install -q git+https://github.com/oliverguhr/deepmultilingualpunctuation.git
!pip install -q yt-dlp
!pip uninstall -y nvidia-cudnn-cu12 2>/dev/null

In [ ]:
# Set your HuggingFace token (required for Pyannote 3.1)
import os
from google.colab import userdata

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab Secrets")
except Exception:
    os.environ["HF_TOKEN"] = ""  # <-- paste your token here if not using Secrets
    if not os.environ["HF_TOKEN"]:
        print("WARNING: HF_TOKEN not set. Pyannote will fail.")
    else:
        print("HF_TOKEN set manually")

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Input Audio

Paste a YouTube URL and it will be downloaded automatically.

In [ ]:
# Paste your YouTube URL here
YOUTUBE_URL = "https://www.youtube.com/watch?v=REPLACE_ME"

# Download audio + auto-captions
import subprocess, re, json as _json

# Extract video ID
vid_match = re.search(r'(?:v=|/)([a-zA-Z0-9_-]{11})', YOUTUBE_URL)
video_id = vid_match.group(1) if vid_match else "audio"
audio_path = f"{video_id}.wav"

# Download audio as WAV 16kHz mono
!yt-dlp -x --audio-format wav --postprocessor-args "ffmpeg:-ar 16000 -ac 1" \
    -o "{video_id}.%(ext)s" "{YOUTUBE_URL}"

# Download auto-captions (English) for accuracy comparison
!yt-dlp --write-auto-sub --sub-lang en --sub-format json3 --skip-download \
    -o "{video_id}" "{YOUTUBE_URL}" 2>/dev/null || echo 'No auto-captions available'

caption_path = f"{video_id}.en.json3"
has_captions = os.path.exists(caption_path)
print(f"\nAudio: {audio_path}")
print(f"YouTube captions available: {has_captions}")

## 3. Run Comparison Pipeline

In [ ]:
import os, sys

WHISPER_DIR = "/content/vault/vendor/whisper-diarization"
os.chdir(WHISPER_DIR)
sys.path.insert(0, WHISPER_DIR)

import nltk
nltk.download("punkt_tab", quiet=True)

from diarize import (
    run_whisper_transcription,
    run_diarization,
    run_postprocessing,
    save_whisper_cache,
)
import time
import json

# Best accuracy settings
whisper_model = "large-v3"
batch_size = 16
device = "cuda"

print("Imports OK")

### Phase A: Whisper Transcription (shared, run once)

In [ ]:
t0 = time.time()
whisper_result = run_whisper_transcription(
    audio_path, whisper_model, device, batch_size, "en",
)
whisper_time = time.time() - t0

cache_path = save_whisper_cache(whisper_result, "whisper_cache")

print(f"\nWhisper completed in {whisper_time:.1f}s")
print(f"  Words: {len(whisper_result['word_timestamps'])}")
print(f"  Duration: {whisper_result['audio_duration']:.0f}s")

### Phase B1: NeMo MSDD

In [ ]:
t0 = time.time()
msdd_speaker_ts = run_diarization(whisper_result["audio_waveform"], "msdd", device)
msdd_diarize_time = time.time() - t0

t0 = time.time()
msdd_segments = run_postprocessing(
    whisper_result["word_timestamps"], msdd_speaker_ts,
    whisper_result["language"], audio_path,
)
msdd_post_time = time.time() - t0

msdd_speakers = {s["speaker"] for s in msdd_segments}
print(f"MSDD: {len(msdd_speakers)} speakers, {len(msdd_segments)} segments, {msdd_diarize_time:.1f}s")

### Phase B2: Pyannote 3.1

In [ ]:
t0 = time.time()
pyannote_speaker_ts = run_diarization(whisper_result["audio_waveform"], "pyannote", device)
pyannote_diarize_time = time.time() - t0

t0 = time.time()
pyannote_segments = run_postprocessing(
    whisper_result["word_timestamps"], pyannote_speaker_ts,
    whisper_result["language"], audio_path,
)
pyannote_post_time = time.time() - t0

pyannote_speakers = {s["speaker"] for s in pyannote_segments}
print(f"Pyannote: {len(pyannote_speakers)} speakers, {len(pyannote_segments)} segments, {pyannote_diarize_time:.1f}s")

## 4. Comparison Summary

In [ ]:
print(f"{'='*70}")
print("COMPARISON SUMMARY")
print(f"{'='*70}")
print(f"Audio: {audio_path} ({whisper_result['audio_duration']:.0f}s)")
print(f"Whisper: {whisper_model} ({whisper_time:.1f}s, shared)")
print()

header = f"{'Metric':<30} {'NeMo MSDD':>15} {'Pyannote 3.1':>15}"
print(header)
print("-" * len(header))
print(f"{'Diarization time':<30} {msdd_diarize_time:>14.1f}s {pyannote_diarize_time:>14.1f}s")
print(f"{'Post-processing time':<30} {msdd_post_time:>14.1f}s {pyannote_post_time:>14.1f}s")
print(f"{'Total time':<30} {msdd_diarize_time+msdd_post_time:>14.1f}s {pyannote_diarize_time+pyannote_post_time:>14.1f}s")
print(f"{'Speakers detected':<30} {len(msdd_speakers):>15} {len(pyannote_speakers):>15}")
print(f"{'Speaker turns (raw)':<30} {len(msdd_speaker_ts):>15} {len(pyannote_speaker_ts):>15}")
print(f"{'Output segments':<30} {len(msdd_segments):>15} {len(pyannote_segments):>15}")

## 5. Accuracy Analysis

### 5a. Speaker Assignment Agreement

Both diarizers assigned speakers to the same set of Whisper words.
We measure what % of words they agree on (after optimal label permutation matching).

In [ ]:
from itertools import permutations
from helpers import get_words_speaker_mapping

# Get word-level speaker assignments from both
msdd_wsm = get_words_speaker_mapping(whisper_result["word_timestamps"], msdd_speaker_ts, "start")
pyannote_wsm = get_words_speaker_mapping(whisper_result["word_timestamps"], pyannote_speaker_ts, "start")

# Extract speaker labels per word
msdd_labels = [w["speaker"] for w in msdd_wsm]
pyannote_labels = [w["speaker"] for w in pyannote_wsm]

# Find best label permutation (Hungarian matching)
msdd_unique = sorted(set(msdd_labels))
pyannote_unique = sorted(set(pyannote_labels))

best_agreement = 0
best_mapping = {}

# Try all permutations of pyannote labels mapped to msdd labels
# (only practical for <=8 speakers)
n = max(len(msdd_unique), len(pyannote_unique))
padded_msdd = msdd_unique + list(range(100, 100 + n - len(msdd_unique)))

for perm in permutations(padded_msdd[:n]):
    mapping = dict(zip(pyannote_unique, perm))
    mapped = [mapping.get(l, -1) for l in pyannote_labels]
    agreement = sum(1 for a, b in zip(msdd_labels, mapped) if a == b)
    if agreement > best_agreement:
        best_agreement = agreement
        best_mapping = mapping

total_words = len(msdd_labels)
agreement_pct = (best_agreement / total_words * 100) if total_words > 0 else 0
disagreement_pct = 100 - agreement_pct

print(f"Word-level speaker assignment comparison:")
print(f"  Total words: {total_words}")
print(f"  Agreement:   {best_agreement}/{total_words} ({agreement_pct:.1f}%)")
print(f"  Disagreement: {total_words - best_agreement}/{total_words} ({disagreement_pct:.1f}%)")
print(f"\n  Best label mapping (pyannote -> msdd): {best_mapping}")
print(f"  MSDD speakers: {msdd_unique}")
print(f"  Pyannote speakers: {pyannote_unique}")

if agreement_pct > 90:
    print(f"\n  HIGH AGREEMENT ({agreement_pct:.1f}%) — both diarizers largely agree on speaker assignments")
elif agreement_pct > 70:
    print(f"\n  MODERATE AGREEMENT ({agreement_pct:.1f}%) — notable differences in speaker boundary placement")
else:
    print(f"\n  LOW AGREEMENT ({agreement_pct:.1f}%) — significant divergence, review segments manually")

### 5b. Disagreement Analysis

Show where the two diarizers disagree — these are the interesting regions.

In [ ]:
# Find disagreement regions
mapped_pyannote = [best_mapping.get(l, -1) for l in pyannote_labels]

disagreements = []
i = 0
while i < total_words:
    if msdd_labels[i] != mapped_pyannote[i]:
        # Start of a disagreement region
        start_i = i
        while i < total_words and msdd_labels[i] != mapped_pyannote[i]:
            i += 1
        end_i = i - 1
        words_in_region = [msdd_wsm[j]["word"] for j in range(start_i, end_i + 1)]
        disagreements.append({
            "start_time": msdd_wsm[start_i]["start_time"] / 1000,
            "end_time": msdd_wsm[end_i]["end_time"] / 1000,
            "n_words": end_i - start_i + 1,
            "msdd_speaker": msdd_labels[start_i],
            "pyannote_speaker": pyannote_labels[start_i],
            "text": " ".join(words_in_region[:15]),
        })
    else:
        i += 1

print(f"Found {len(disagreements)} disagreement regions\n")
print(f"{'Time Range':<20} {'Words':>6} {'MSDD':>8} {'Pyannote':>10} Text")
print("-" * 90)
for d in disagreements[:20]:
    time_range = f"{d['start_time']:.1f}s-{d['end_time']:.1f}s"
    print(f"{time_range:<20} {d['n_words']:>6} {'Spk '+str(d['msdd_speaker']):>8} {'Spk '+str(d['pyannote_speaker']):>10} {d['text'][:50]}")

if len(disagreements) > 20:
    print(f"\n... and {len(disagreements) - 20} more disagreement regions")

### 5c. YouTube Caption Comparison (if available)

Compare both diarizers' word-level text against YouTube's auto-captions to estimate WER.

In [ ]:
if has_captions:
    # Parse YouTube json3 captions
    with open(caption_path, "r") as f:
        yt_data = json.load(f)

    yt_words = []
    for event in yt_data.get("events", []):
        if "segs" in event:
            base_ms = event.get("tStartMs", 0)
            for seg in event["segs"]:
                text = seg.get("utf8", "").strip()
                if text and text != "\n":
                    offset_ms = seg.get("tOffsetMs", 0)
                    yt_words.append({
                        "text": text.lower(),
                        "start_ms": base_ms + offset_ms,
                    })

    # Compare Whisper transcript against YouTube captions
    whisper_text = " ".join(w["text"].lower() for w in whisper_result["word_timestamps"])
    yt_text = " ".join(w["text"] for w in yt_words)

    # Simple word-level comparison
    whisper_words_list = whisper_text.split()
    yt_words_list = yt_text.split()

    # Compute overlap (not true WER, but a reasonable proxy)
    whisper_set = set(enumerate(whisper_words_list))
    common = len(set(whisper_words_list) & set(yt_words_list))
    total_unique = len(set(whisper_words_list) | set(yt_words_list))
    vocabulary_overlap = common / total_unique * 100 if total_unique > 0 else 0

    print(f"YouTube Caption Comparison:")
    print(f"  Whisper words: {len(whisper_words_list)}")
    print(f"  YouTube words: {len(yt_words_list)}")
    print(f"  Vocabulary overlap: {vocabulary_overlap:.1f}%")
    print(f"\n  Note: Both diarizers use the same Whisper transcript.")
    print(f"  The difference is only in SPEAKER assignment, not in text accuracy.")
    print(f"  Text accuracy (Whisper vs YouTube) is shared between both pipelines.")
else:
    print("No YouTube captions available for this video.")
    print("Speaker assignment accuracy can only be compared between the two diarizers (see 5a above).")

## 6. Side-by-Side Transcript

In [ ]:
print("--- NeMo MSDD (first 10 segments) ---")
for seg in msdd_segments[:10]:
    print(f"  [{seg['start']:7.1f}s - {seg['end']:7.1f}s] {seg['speaker']}: {seg['text'][:80]}")

print("\n--- Pyannote 3.1 (first 10 segments) ---")
for seg in pyannote_segments[:10]:
    print(f"  [{seg['start']:7.1f}s - {seg['end']:7.1f}s] {seg['speaker']}: {seg['text'][:80]}")

## 7. Save & Download Results

In [ ]:
# Save all results
base = audio_path.rsplit('.', 1)[0]

with open(f"{base}_msdd.json", "w") as f:
    json.dump(msdd_segments, f, ensure_ascii=False, indent=2)

with open(f"{base}_pyannote.json", "w") as f:
    json.dump(pyannote_segments, f, ensure_ascii=False, indent=2)

benchmark = {
    "video_id": video_id,
    "audio_duration_s": round(whisper_result["audio_duration"], 1),
    "whisper_model": whisper_model,
    "whisper_time_s": round(whisper_time, 1),
    "speaker_agreement_pct": round(agreement_pct, 1),
    "n_disagreement_regions": len(disagreements),
    "nemo_msdd": {
        "diarize_time_s": round(msdd_diarize_time, 1),
        "n_speakers": len(msdd_speakers),
        "n_segments": len(msdd_segments),
        "n_speaker_turns": len(msdd_speaker_ts),
    },
    "pyannote_3_1": {
        "diarize_time_s": round(pyannote_diarize_time, 1),
        "n_speakers": len(pyannote_speakers),
        "n_segments": len(pyannote_segments),
        "n_speaker_turns": len(pyannote_speaker_ts),
    },
}

with open(f"{base}_benchmark.json", "w") as f:
    json.dump(benchmark, f, ensure_ascii=False, indent=2)

print("Saved:")
print(f"  {base}_msdd.json")
print(f"  {base}_pyannote.json")
print(f"  {base}_benchmark.json")

In [ ]:
from google.colab import files

for path in [f"{base}_msdd.json", f"{base}_pyannote.json", f"{base}_benchmark.json"]:
    if os.path.exists(path):
        files.download(path)